# ⚛️ Week 5 — Noise & the NISQ Era
**Track:** Undergraduate | **Lab:** Quantum Learning + TNBC Lab

---

## Overview
Real quantum hardware is noisy. This notebook introduces quantum noise models, the NISQ (Noisy Intermediate-Scale Quantum) era, and error mitigation strategies relevant to near-term quantum machine learning applications like TNBC classification.

**Learning Objectives:**
- Define decoherence, gate error, and measurement error
- Simulate noise with Qiskit Aer's noise models
- Compare noiseless vs. noisy circuit outputs
- Understand error mitigation (readout correction, ZNE)

## 📚 Concept Review

### What is NISQ?
NISQ = **Noisy Intermediate-Scale Quantum** (Preskill, 2018)
- **50–1000 qubits** (current hardware: 100s to ~1000)
- **No fault-tolerant error correction** — errors accumulate
- **Short circuit depth** required before decoherence ruins results
- Hybrid quantum-classical algorithms (VQE, QAOA, QNNs) are designed for this era

### Main Noise Sources
| Type | Description | Effect |
|------|-------------|--------|
| Depolarizing | Random Pauli error | Pushes state toward mixed |
| Bit-flip | X applied randomly | $\|0\rangle \leftrightarrow \|1\rangle$ |
| Phase-flip | Z applied randomly | Destroys phase info |
| Amplitude damping | Energy loss (T1) | $\|1\rangle \to \|0\rangle$ |
| Measurement | Readout misclassification | Wrong bit recorded |

## 🔧 Environment Setup

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
import numpy as np
print('Qiskit Aer noise tools ready ✓')

## 🔵 Section 1 — Noiseless Baseline

In [ ]:
# Perfect Bell state: should give 50% |00⟩ and 50% |11⟩
qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0,1],[0,1])

sim_ideal = AerSimulator()
counts_ideal = sim_ideal.run(qc, shots=4096).result().get_counts()
print(f'Ideal counts: {counts_ideal}')

## 🔴 Section 2 — Adding Depolarizing Noise

In [ ]:
# Build a noise model with depolarizing errors
noise_model = NoiseModel()

# 1% gate error on single-qubit gates
error_1q = depolarizing_error(0.01, 1)
noise_model.add_all_qubit_quantum_error(error_1q, ['h', 'x', 'rz', 's', 't'])

# 3% gate error on two-qubit gates
error_2q = depolarizing_error(0.03, 2)
noise_model.add_all_qubit_quantum_error(error_2q, ['cx'])

print(noise_model)

In [ ]:
sim_noisy = AerSimulator(noise_model=noise_model)
counts_noisy = sim_noisy.run(qc, shots=4096).result().get_counts()
print(f'Noisy counts: {counts_noisy}')

In [ ]:
# Compare ideal vs noisy
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_histogram(counts_ideal, ax=axes[0], title='Ideal (No Noise)', color='steelblue')
plot_histogram(counts_noisy,  ax=axes[1], title='Noisy (Depolarizing 1%/3%)', color='salmon')
plt.tight_layout()
plt.show()

## 📉 Section 3 — Noise Scaling (Circuit Depth Effect)

In [ ]:
# Observe how noise degrades fidelity as circuit depth increases
n_cx_gates = [1, 3, 5, 10, 20]
counts_00 = []

for n_cx in n_cx_gates:
    qc_deep = QuantumCircuit(2, 2)
    qc_deep.h(0)
    for _ in range(n_cx):
        qc_deep.cx(0, 1)
        qc_deep.cx(1, 0)  # cancel out but adds noise
    qc_deep.measure([0,1],[0,1])
    counts = sim_noisy.run(qc_deep, shots=2048).result().get_counts()
    p00 = counts.get('00', 0) / 2048
    counts_00.append(p00)

plt.figure(figsize=(8, 4))
plt.plot(n_cx_gates, counts_00, 'o-', color='crimson', linewidth=2)
plt.axhline(0.5, linestyle='--', color='gray', label='Ideal P(|00⟩)=0.5')
plt.xlabel('Number of CX gate pairs')
plt.ylabel('P(|00⟩)')
plt.title('Noise Accumulation vs Circuit Depth')
plt.legend()
plt.tight_layout()
plt.show()

## 🛡️ Section 4 — Readout Error Mitigation

In [ ]:
# Add readout error
readout_error = ReadoutError([[0.95, 0.05], [0.08, 0.92]])  # 5% and 8% flip rates
noise_model_ro = NoiseModel()
noise_model_ro.add_all_qubit_readout_error(readout_error)

sim_ro = AerSimulator(noise_model=noise_model_ro)

# Apply to a simple |0⟩ circuit (should measure all 0s)
qc_simple = QuantumCircuit(1, 1)
qc_simple.measure(0, 0)
counts_ro = sim_ro.run(qc_simple, shots=4096).result().get_counts()
print(f'|0⟩ with readout error: {counts_ro}')
print(f'Expected "1" fraction (noise): {counts_ro.get("1",0)/4096:.3f} (ideal: 0.000)')

## 🎯 Exercise 1
Increase the depolarizing error to 5% for single-qubit gates. How much does the Bell state fidelity degrade?

In [ ]:
# ── TODO ──
# noise_model_5pct = NoiseModel()
# error_5pct = depolarizing_error(0.05, 1)
# noise_model_5pct.add_all_qubit_quantum_error(error_5pct, ['h', 'x'])
# sim_5pct = AerSimulator(noise_model=noise_model_5pct)
# counts_5pct = sim_5pct.run(qc, shots=4096).result().get_counts()
# print(counts_5pct)

## 🎯 Exercise 2
Research one real NISQ device (IBM, IonQ, or Quantinuum) and record its:
- Number of qubits
- T1 / T2 coherence times
- 2-qubit gate fidelity
- Quantum volume

In [ ]:
# ── TODO: Fill in from device documentation ──
device_specs = {
    'device_name': '',      # e.g., 'ibm_kyoto'
    'n_qubits':     None,
    'T1_us':        None,   # microseconds
    'T2_us':        None,
    'cx_fidelity':  None,   # e.g., 0.997
    'quantum_volume': None
}
print(device_specs)

## 💡 Key Takeaways
- Noise is the central challenge of NISQ computing — it limits circuit depth
- **Depolarizing** noise drives states toward uniform mixtures
- **Readout error** misclassifies measurement outcomes
- Near-term QML must use shallow, noise-resilient circuits
- Error **mitigation** (not correction) is the practical NISQ approach

## 📝 TODO Checklist
- [ ] Complete Exercise 1: 5% depolarizing noise Bell fidelity
- [ ] Complete Exercise 2: Real device specs research
- [ ] Explore amplitude damping noise model
- [ ] Plot T1 decay curve for a |1⟩ state with amplitude damping

---
*Quantum Learning + TNBC Lab | Undergraduate Track | Week 5*